Первым делом скачаем все необходимые зависимости

In [2]:
!pip install stanza
!pip install allennlp==2.10.1 allennlp-models==2.10.1
!pip install torch
!pip install pandas
!pip install scikit-learn
!pip install matplotlib seaborn

     ---------------------------------------- 1.7/1.7 MB 3.9 MB/s eta 0:00:00
     -------------------------------------- 241.2/241.2 MB 5.9 MB/s eta 0:00:00
     -------------------------------------- 608.4/608.4 KB 9.6 MB/s eta 0:00:00
     ---------------------------------------- 1.7/1.7 MB 9.9 MB/s eta 0:00:00
     ---------------------------------------- 1.6/1.6 MB 9.5 MB/s eta 0:00:00
  Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
     ---------------------------------------- 6.3/6.3 MB 9.6 MB/s eta 0:00:00
     ------------------------------------- 536.2/536.2 KB 11.2 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.5.0
    Uninstalling typing_extensions-4.5.0:
      Successfully uninstalled typing_extensions-4.5.0
  Attempting uninstall: torch
    Found existing installation: torch 1.12.1
    Uninstalling torch-1.12.1:
      Successfully uninstalled torch-1.12.1


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.13.1 requires torch==1.12.1, but you have torch 2.8.0 which is incompatible.
spacy 3.3.3 requires typing-extensions<4.6.0,>=3.7.4.1, but you have typing-extensions 4.15.0 which is incompatible.
allennlp 2.10.1 requires torch<1.13.0,>=1.10.0, but you have torch 2.8.0 which is incompatible.
allennlp-models 2.10.1 requires torch<1.13.0,>=1.7.0, but you have torch 2.8.0 which is incompatible.
You should consider upgrading via the 'C:\Users\admin\Desktop\case_study\case_study\venv39\Scripts\python.exe -m pip install --upgrade pip' command.


  Using cached torch-1.12.1-cp39-cp39-win_amd64.whl (161.8 MB)
  Using cached typing_extensions-4.5.0-py3-none-any.whl (27 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: torch
    Found existing installation: torch 2.8.0
    Uninstalling torch-2.8.0:
      Successfully uninstalled torch-2.8.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
stanza 1.11.0 requires torch>=1.13.0, but you have torch 1.12.1 which is incompatible.
You should consider upgrading via the 'C:\Users\admin\Desktop\case_study\case_study\venv39\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\admin\Desktop\case_study\case_study\venv39\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\admin\Desktop\case_study\case_study\venv39\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\admin\Desktop\case_study\case_study\venv39\Scripts\python.exe -m pip install --upgrade pip' command.


You should consider upgrading via the 'C:\Users\admin\Desktop\case_study\case_study\venv39\Scripts\python.exe -m pip install --upgrade pip' command.


In [2]:
import torch
import random
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

In [6]:
import stanza

stanza.download('en')

nlp = stanza.Pipeline(
    'en',
    processors='tokenize,pos,lemma,depparse,constituency'
)

sentence = "John gave Mary a book."
doc = nlp(sentence)

for sent in doc.sentences:
    for word in sent.words:
        print(word.text, word.deprel, word.head)

2026-04-19 22:40:52 INFO: Downloaded file to C:\Users\admin\stanza_resources\resources.json
2026-04-19 22:40:52 INFO: Downloading default packages for language: en (English) ...
2026-04-19 22:40:54 INFO: File exists: C:\Users\admin\stanza_resources\en\default.zip
2026-04-19 22:40:58 INFO: Finished downloading models and saved to C:\Users\admin\stanza_resources
2026-04-19 22:40:58 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-04-19 22:40:59 INFO: Downloaded file to C:\Users\admin\stanza_resources\resources.json
2026-04-19 22:40:59 WARNING: Language en package default expects mwt, which has been added
2026-04-19 22:41:00 INFO: Loading these models for language: en (English):
| Processor    | Package             |
--------------------------------------
| tokenize     | combined            |
| mwt          | combined            |
| pos    

John nsubj 2
gave root 0
Mary iobj 2
a det 5
book obj 2
. punct 2


In [7]:
def simple_srl_from_doc(doc):
    roles = []

    for sent in doc.sentences:
        for word in sent.words:
            if word.upos == "VERB":
                predicate = word.text
                args = {"V": predicate}

                for w in sent.words:
                    if w.head == word.id:
                        if w.deprel == "nsubj":
                            args["ARG0"] = w.text
                        elif w.deprel in ["obj", "dobj"]:
                            args["ARG1"] = w.text
                        elif w.deprel == "iobj":
                            args["ARG2"] = w.text

                roles.append(args)

    return roles

In [8]:
sentence = "John gave Mary a book."
doc = nlp(sentence)

baseline_roles = simple_srl_from_doc(doc)
print(baseline_roles)

[{'V': 'gave', 'ARG0': 'John', 'ARG2': 'Mary', 'ARG1': 'book'}]


In [9]:
import random
import copy

def corrupt_dependencies(doc, corruption_rate=0.3):
    corrupted_doc = copy.deepcopy(doc)

    for sent in corrupted_doc.sentences:
        words = sent.words
        for word in words:
            if random.random() < corruption_rate and word.head != 0:
                # случайно выбираем новый head
                possible_heads = [w.id for w in words if w.id != word.id]
                word.head = random.choice(possible_heads)

    return corrupted_doc

In [10]:
corrupted_doc = corrupt_dependencies(doc, corruption_rate=0.5)

corrupted_roles = simple_srl_from_doc(corrupted_doc)

print("Baseline:", baseline_roles)
print("Corrupted:", corrupted_roles)

Baseline: [{'V': 'gave', 'ARG0': 'John', 'ARG2': 'Mary', 'ARG1': 'book'}]
Corrupted: [{'V': 'gave', 'ARG1': 'book'}]
